# CLIP Image Auditor — Optimized

**Fast mode:** Copies images to Colab's local SSD first (10x faster reads), then classifies on GPU.

### Setup
1. Set `INPUT_DIR` to your image folder on Drive
2. Set `OUTPUT_DIR` where sorted results go
3. Click **Runtime → Run all**

In [ ]:
# @title Step 1: Install + Mount Drive
!pip install -q git+https://github.com/openai/CLIP.git pandas

from google.colab import drive
drive.mount('/content/drive')

# @markdown **Set these paths:**
INPUT_DIR = '/content/drive/MyDrive/batch_001/part_001/files'  # @param {type:"string"}
OUTPUT_DIR = '/content/drive/MyDrive/audited'  # @param {type:"string"}

# Verify input exists
import os
assert os.path.isdir(INPUT_DIR), f"Not found: {INPUT_DIR}"
pngs = [f for f in os.listdir(INPUT_DIR) if f.endswith('.png')]
print(f"✅ Found {len(pngs)} images in: {INPUT_DIR}")

In [ ]:
# @title Step 2: Copy images to local SSD (10x faster reads)
import shutil

LOCAL_DIR = '/content/local_images'
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
os.makedirs(LOCAL_DIR)

print(f"Copying {len(pngs)} images to local SSD...")
for f in pngs:
    shutil.copy2(os.path.join(INPUT_DIR, f), os.path.join(LOCAL_DIR, f))

print(f"✅ Copied {len(pngs)} images to {LOCAL_DIR}")
print(f"   Size: {sum(os.path.getsize(os.path.join(LOCAL_DIR, f)) for f in pngs) / 1024 / 1024:.1f} MB")

In [ ]:
# @title Step 3: Load CLIP + Define Prompts
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

GRAPHICS_PROMPTS = [
    "a chart with axes and data points",
    "a bar chart showing data",
    "a pie chart with colored segments",
    "a line graph with trends",
    "a map with geographic data",
    "an infographic with visual data",
    "a diagram showing a process or flow",
    "a scatter plot with data distribution",
    "a data visualization dashboard",
    "a technical drawing or schematic",
    "a heatmap showing data patterns",
    "a funnel chart or waterfall chart",
]

NON_GRAPHICS_PROMPTS = [
    "a slide with bullet points and text",
    "a photograph of people or scenery",
    "a title slide with large text",
    "a table with rows and columns of text",
    "a page with mostly text content",
    "a decorative image or logo",
    "a closing slide with contact information",
    "a photo of a presentation speaker",
    "a slide with quotes or testimonials",
    "a slide with company branding",
    "a white background with small text",
    "a screenshot of a software interface",
]

graphics_tokens = clip.tokenize(GRAPHICS_PROMPTS).to(device)
non_graphics_tokens = clip.tokenize(NON_GRAPHICS_PROMPTS).to(device)

with torch.no_grad():
    gf = model.encode_text(graphics_tokens).mean(dim=0)
    nf = model.encode_text(non_graphics_tokens).mean(dim=0)
    gf = gf / gf.norm()
    nf = nf / nf.norm()

print(f"✅ CLIP loaded on {device} | {len(GRAPHICS_PROMPTS)}+{len(NON_GRAPHICS_PROMPTS)} prompts")

In [ ]:
# @title Step 4: Batch Classify All Images
import torch.nn.functional as F
import time

BATCH_SIZE = 128  # @param {type:"integer"}
PASS_THRESH = 0.70  # @param {type:"number"}
REJECT_THRESH = 0.40  # @param {type:"number"}

all_files = sorted([f for f in os.listdir(LOCAL_DIR) if f.endswith('.png')])
print(f"🔍 Classifying {len(all_files)} images (batch={BATCH_SIZE}) on {device}\n")

results = []
t0 = time.time()

for i in range(0, len(all_files), BATCH_SIZE):
    batch = all_files[i:i+BATCH_SIZE]
    images = []
    valid = []
    for f in batch:
        try:
            images.append(preprocess(Image.open(os.path.join(LOCAL_DIR, f)).convert('RGB')))
            valid.append(f)
        except:
            results.append({'filename': f, 'graphics_score': 0, 'classification': 'error', 'reasons': 'load_failed'})

    if not images:
        continue

    batch_tensor = torch.stack(images).to(device)
    with torch.no_grad():
        feats = model.encode_image(batch_tensor)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        gs = (feats @ gf).cpu()
        ns = (feats @ nf).cpu()
        probs = F.softmax(torch.stack([gs, ns], dim=-1) * 100, dim=-1)

    for j, f in enumerate(valid):
        gp = probs[j, 0].item()
        if gp > PASS_THRESH:
            cls, reason = 'pass', 'high_confidence' if gp > 0.90 else 'moderate_confidence'
        elif gp > REJECT_THRESH:
            cls, reason = 'borderline', 'uncertain'
        else:
            cls, reason = 'reject', 'likely_non_graphics'
        results.append({'filename': f, 'graphics_score': round(gp, 4), 'classification': cls, 'reasons': reason})

    done = len(results)
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed else 0
    eta = (len(all_files) - done) / rate if rate else 0
    p = sum(1 for r in results[-len(batch):] if r['classification']=='pass')
    r_ = sum(1 for r in results[-len(batch):] if r['classification']=='reject')
    b = sum(1 for r in results[-len(batch):] if r['classification']=='borderline')
    print(f"  [{done:>5}/{len(all_files)}] pass={p} reject={r_} borderline={b} | {rate:.0f} img/sec | ETA: {eta:.0f}s")

print(f"\n✅ Done: {len(results)} images in {time.time()-t0:.1f}s ({len(results)/(time.time()-t0):.0f} img/sec)")

In [ ]:
# @title Step 5: Sort + Save to Drive
import csv, json, shutil

pass_list = [r for r in results if r['classification']=='pass']
reject_list = [r for r in results if r['classification']=='reject']
border_list = [r for r in results if r['classification']=='borderline']

for folder, lst in [('PASS', pass_list), ('REJECT', reject_list), ('BORDERLINE', border_list)]:
    d = os.path.join(OUTPUT_DIR, folder)
    os.makedirs(d, exist_ok=True)
    for r in lst:
        src = os.path.join(LOCAL_DIR, r['filename'])
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(d, r['filename']))
    # Write audit CSV
    with open(os.path.join(d, 'audit_log.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['filename','classification','graphics_score','reasons'])
        for r in sorted(lst, key=lambda x: x.get('graphics_score',0), reverse=True):
            w.writerow([r['filename'], r['classification'], r['graphics_score'], r['reasons']])

# Summary
summary = {
    'total': len(results), 'pass': len(pass_list), 'reject': len(reject_list),
    'borderline': len(border_list),
    'pass_pct': round(len(pass_list)/max(len(results),1)*100,1),
    'reject_pct': round(len(reject_list)/max(len(results),1)*100,1),
    'borderline_pct': round(len(border_list)/max(len(results),1)*100,1),
}
with open(os.path.join(OUTPUT_DIR, 'audit_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("=" * 55)
print("  AUDIT COMPLETE")
print("=" * 55)
print(f"  Total:      {summary['total']}")
print(f"  PASS:       {summary['pass']} ({summary['pass_pct']}%)")
print(f"  BORDERLINE: {summary['borderline']} ({summary['borderline_pct']}%)")
print(f"  REJECT:     {summary['reject']} ({summary['reject_pct']}%)")
print("=" * 55)
print(f"  📁 {OUTPUT_DIR}/")
print(f"     PASS/         — {len(pass_list)} images + audit_log.csv")
print(f"     REJECT/       — {len(reject_list)} images + audit_log.csv")
print(f"     BORDERLINE/   — {len(border_list)} images + audit_log.csv")
print(f"     audit_summary.json")

# Cleanup local copy
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
print(f"\n🧹 Cleaned up local cache")